In [1]:
import os
import chardet
import pandas as pd

In [2]:
def detect_encoding(file_path):
    """
    Detect the encoding of a text file.
    
    Args:
        file_path (str): Path to the file.
    
    Returns:
        str: Detected encoding.
    """
    with open(file_path, 'rb') as f:
        raw_data = f.read(10000)
        result = chardet.detect(raw_data)
        return result['encoding']

def collect_txt_to_csv(root_folder, output_csv):
    """
    Collect all .txt files from subdirectories of `root_folder`, 
    and save them into a CSV file with columns: text, label.

    Args:
        root_folder (str): Path to the root folder containing category subfolders.
        output_csv (str): Path to the output CSV file.
    """
    data = []
    
    for label in os.listdir(root_folder):
        label_path = os.path.join(root_folder, label)
        if os.path.isdir(label_path):  # Only process directories
            for filename in os.listdir(label_path):
                if filename.endswith('.txt'):
                    file_path = os.path.join(label_path, filename)
                    try:
                        # Auto-detect encoding
                        encoding = detect_encoding(file_path)
                        if encoding is None:
                            print(f"Warning: Could not detect encoding for {file_path}, skipping...")
                            continue

                        # Read the file with detected encoding
                        with open(file_path, 'r', encoding=encoding, errors='replace') as file:
                            text = file.read().strip()  # Read and remove extra whitespace
                        
                        data.append({'text': text, 'label': label})
                    except Exception as e:
                        print(f"Error reading file {file_path}: {e}")

    # Convert to DataFrame
    df = pd.DataFrame(data)

    # Save as CSV with UTF-8-SIG encoding
    df.to_csv(output_csv, index=False, encoding='utf-8-sig')
    print(f"Successfully saved {len(df)} records to {output_csv}") 

In [3]:
train_path = 'dataset/Train_Full'
train_output = 'train.csv'
test_path = 'dataset/Test_Full'
test_output = 'test.csv'

collect_txt_to_csv(root_folder=train_path, output_csv=train_output)
collect_txt_to_csv(root_folder=test_path, output_csv=test_output)

Successfully saved 33759 records to train.csv
Successfully saved 50373 records to test.csv
